# Smart Ensembling — End-to-End Workflow

Full pipeline:
1. Install required libraries
2. Import libraries
3. Discover & pair OOF / test-probability paths (saved to JSON manifest)
4. Load data with row-count validation against train.csv / test.csv
5. Validate probability bounds [0, 1]
6. Hill-climbing greedy blend with PyTorch (CUDA/CPU auto-detect)


## Step 1 — Install Required Libraries


In [1]:
# Install / upgrade all dependencies needed for the ensembling workflow.
# Re-run this cell if any import fails below.
import subprocess, sys

packages = [
    "numpy>=1.26",
    "pandas>=2.0",
    "scikit-learn>=1.4",
    "torch>=2.2"
]

for pkg in packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", pkg]
    )
    print(f"[OK] {pkg}")

# PyTorch — try CUDA wheel first, fall back to CPU-only
try:
    import torch
    print(f"[OK] torch {torch.__version__} already installed")
except ImportError:
    print("torch not found — installing CPU wheel (replace index-url for CUDA)")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "torch", "--index-url", "https://download.pytorch.org/whl/cu121"
    ])
    print("[OK] torch installed")

print("\nAll libraries ready.")


[OK] numpy>=1.26
[OK] pandas>=2.0
[OK] scikit-learn>=1.4
[OK] torch>=2.2
[OK] torch 2.13.0+cpu already installed

All libraries ready.


## Step 2 — Import Libraries


In [2]:
from __future__ import annotations

import json
import time
from pathlib import Path
from typing import Any
from datetime import datetime
import numpy as np
import pandas as pd
import torch
import copy
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

print(f"numpy  : {np.__version__}")
print(f"pandas : {pd.__version__}")
print(f"torch  : {torch.__version__}")

# ── Device selection ──────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nPyTorch device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU          : {torch.cuda.get_device_name(0)}")

numpy  : 2.5.2
pandas : 3.0.5
torch  : 2.13.0+cpu

PyTorch device : cpu


## Step 3 — Discover & Pair OOF / Test-Probability Paths

Recursively scans `oof_&_probs/` and builds a dict  
`{ relative_oof_path : relative_test_path }`.  
Supports two naming conventions:
- `oof_<name>.npy` ↔ `test_<name>.npy`
- `<name>_oof.npy` ↔ `<name>_test.npy`

The manifest is written to `ensemble_artifacts/paired_prediction_paths.json`.


In [3]:
# ── Configuration (edit paths here if needed) ─────────────────────────────────────
PREDICTION_ROOT = Path("oof_&_probs")
TRAIN_CSV       = Path("input_data/train.csv")
TEST_CSV        = Path("input_data/test.csv")
TARGET_COL      = "addicted_label"
ARTIFACTS_DIR   = Path("ensemble_artifacts")
MANIFEST_PATH   = ARTIFACTS_DIR / "paired_prediction_paths.json"

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


def rel(path: Path, root: Path) -> str:
    return path.relative_to(root).as_posix()


def find_prediction_pairs(root: Path) -> dict:
    # Supports: oof_<name>.npy <-> test_<name>.npy
    #           <name>_oof.npy <-> <name>_test.npy
    pairs = {}
    for oof_path in sorted(root.rglob("*.npy")):
        name = oof_path.name
        if name.startswith("oof_"):
            test_name = f"test_{name[len('oof_'):]}"
        elif name.endswith("_oof.npy"):
            test_name = f"{name[:-len('_oof.npy')]}_test.npy"
        else:
            continue
        test_path = oof_path.with_name(test_name)
        if test_path.is_file():
            pairs[rel(oof_path, root)] = rel(test_path, root)
        else:
            print(
                f"[UNPAIRED] {rel(oof_path, root)} — "
                f"expected companion '{test_name}' not found"
            )
    return pairs


paired_paths = find_prediction_pairs(PREDICTION_ROOT)

MANIFEST_PATH.write_text(
    json.dumps(paired_paths, indent=2, sort_keys=True),
    encoding="utf-8"
)

print(f"\nDiscovered {len(paired_paths)} OOF/test pairs.")
print(f"Manifest saved -> {MANIFEST_PATH}")
print()
for i, (ok, tk) in enumerate(paired_paths.items()):
    print(f"  OOF  : {ok}")
    print(f"  test : {tk}")
    print()
    if i >= 4:
        print(f"  ... ({len(paired_paths) - 5} more pairs)")
        break



Discovered 109 OOF/test pairs.
Manifest saved -> ensemble_artifacts\paired_prediction_paths.json

  OOF  : solution_0/lightgbm_oof.npy
  test : solution_0/lightgbm_test.npy

  OOF  : solution_2/oof_catnative.npy
  test : solution_2/test_catnative.npy

  OOF  : solution_2/oof_gcatd8.npy
  test : solution_2/test_gcatd8.npy

  OOF  : solution_2/oof_gcatlr02.npy
  test : solution_2/test_gcatlr02.npy

  OOF  : solution_2/oof_gcatnote.npy
  test : solution_2/test_gcatnote.npy

  ... (104 more pairs)


## Step 4 — Load Data & Validate Row Counts

* OOF rows must equal `train.csv` rows — mismatches are skipped with a warning.
* Test rows must equal `test.csv` rows — same rule.
* Valid pairs stored in two dicts with the **same key** so they remain permanently paired.


In [4]:
# ── Load reference CSVs ──────────────────────────────────────────────────────────────
train_df = pd.read_csv(TRAIN_CSV, usecols=[TARGET_COL])
test_df  = pd.read_csv(TEST_CSV,  usecols=["id"])

y        = train_df[TARGET_COL].to_numpy()
test_ids = test_df["id"].to_numpy()
n_train  = len(y)
n_test   = len(test_ids)

print(f"train.csv rows : {n_train:,}")
print(f"test.csv rows  : {n_test:,}")

unique_labels = set(np.unique(y).tolist())
if unique_labels - {0, 1}:
    raise ValueError(
        f"Target '{TARGET_COL}' must be binary 0/1; found {unique_labels}"
    )


def load_prob(path: Path) -> np.ndarray:
    # Load a .npy file; squeeze (N,1) -> (N,)
    arr = np.load(path, allow_pickle=False)
    arr = np.asarray(arr)
    if arr.ndim == 2 and arr.shape[1] == 1:
        arr = arr[:, 0]
    if arr.ndim != 1:
        raise ValueError(f"Expected 1-D array, got shape {arr.shape}")
    return arr


# Re-load manifest so this cell is self-contained
paired_paths = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))

oof_dict:  dict = {}
test_dict: dict = {}
skipped = []

for oof_key, test_key in paired_paths.items():
    oof_path  = PREDICTION_ROOT / oof_key
    test_path = PREDICTION_ROOT / test_key
    try:
        oof_arr  = load_prob(oof_path)
        test_arr = load_prob(test_path)
    except Exception as exc:
        print(f"[SKIP — read error] {oof_key} | {exc}")
        skipped.append(oof_key)
        continue
    if len(oof_arr) != n_train:
        print(
            f"[SKIP — row mismatch] {oof_key} | "
            f"OOF has {len(oof_arr):,} rows, expected {n_train:,} (train.csv)"
        )
        skipped.append(oof_key)
        continue
    if len(test_arr) != n_test:
        print(
            f"[SKIP — row mismatch] {oof_key} | "
            f"test prob has {len(test_arr):,} rows, expected {n_test:,} (test.csv)"
        )
        skipped.append(oof_key)
        continue
    oof_dict[oof_key]  = oof_arr.astype(np.float64, copy=False)
    test_dict[oof_key] = test_arr.astype(np.float64, copy=False)

print(f"\nLoaded  : {len(oof_dict):,} valid pairs")
print(f"Skipped : {len(skipped):,} pairs")
if skipped:
    print("Skipped files:")
    for s in skipped:
        print(f"  * {s}")


train.csv rows : 691,369
test.csv rows  : 296,302

Loaded  : 109 valid pairs
Skipped : 0 pairs


## Step 5 — Validate Probability Bounds [0, 1]

Every value must lie in `[0.0, 1.0]`.  
Violating pairs are removed with a printed warning.


In [5]:
def probability_problem(arr, label):
    # Returns a problem string if the array is not a valid probability vector,
    # or None if everything is fine.
    if not np.issubdtype(arr.dtype, np.number):
        return f"{label}: non-numeric dtype '{arr.dtype}'"
    if not np.isfinite(arr).all():
        return f"{label}: contains NaN or +-Inf"
    lo, hi = float(arr.min()), float(arr.max())
    if lo < 0.0 or hi > 1.0:
        return f"{label}: values outside [0, 1] -> min={lo:.8g}, max={hi:.8g}"
    return None


invalid_keys = []
for key in list(oof_dict.keys()):
    problem = (
        probability_problem(oof_dict[key],  f"OOF  '{key}'")
        or probability_problem(test_dict[key], f"test '{key}'")
    )
    if problem:
        print(f"[WARN — prob out of range] {problem}")
        del oof_dict[key]
        del test_dict[key]
        invalid_keys.append(key)

print("\nAfter bounds check:")
print(f"  Valid pairs   : {len(oof_dict):,}")
print(f"  Removed pairs : {len(invalid_keys):,}")

if not oof_dict:
    raise RuntimeError("No valid OOF/test pairs remain after validation.")

print(f"\n{'Model key':<60} {'OOF rows':>10} {'test rows':>10}")
print("-" * 82)
for key in oof_dict:
    print(f"{key:<60} {len(oof_dict[key]):>10,} {len(test_dict[key]):>10,}")


[WARN — prob out of range] test 'solution_2/oof_gxgbd4.npy': values outside [0, 1] -> min=6.2094675e-05, max=1
[WARN — prob out of range] test 'solution_2/oof_gxgbnote.npy': values outside [0, 1] -> min=0.00011063993, max=1
[WARN — prob out of range] test 'solution_3/oof_b.npy': values outside [0, 1] -> min=7.2589849e-06, max=1
[WARN — prob out of range] test 'solution_3/oof_e.npy': values outside [0, 1] -> min=3.8026621e-05, max=1
[WARN — prob out of range] OOF  'solution_4/oof_fmdeep.npy': values outside [0, 1] -> min=-13.279112, max=37.872491
[WARN — prob out of range] OOF  'solution_4/oof_fmnum.npy': values outside [0, 1] -> min=-12.084252, max=40.414283
[WARN — prob out of range] OOF  'solution_4/oof_fmplr.npy': values outside [0, 1] -> min=-14.81429, max=48.67542
[WARN — prob out of range] OOF  'solution_4/oof_fmpure.npy': values outside [0, 1] -> min=-4.9340657, max=40.066786
[WARN — prob out of range] OOF  'solution_4/oof_fmwide.npy': values outside [0, 1] -> min=-9.2629356, ma

## Step 6 — Hill-Climbing Ensemble (PyTorch, CUDA/CPU)

Greedy sequential blend search on OOF predictions:
* Starts from the single best model (highest OOF AUC).
* For every other model, tries blending weights `linspace(0.01, 0.99, 99)` and keeps the best.
* Uses the **exact same logic** as `run_hill_climbing_ensemble.py` and the reference notebook.
* Outputs: blended test probabilities `.npy`, submission CSV, JSON result report.


In [6]:
def binary_roc_auc(y_true, scores):
    # Exact, tie-aware binary ROC-AUC (Wilcoxon-Mann-Whitney).
    # No sklearn dependency required.
    y_true = np.asarray(y_true, dtype=np.int8)
    scores = np.asarray(scores, dtype=np.float64)
    pos = int(y_true.sum())
    neg = len(y_true) - pos
    if pos == 0 or neg == 0:
        raise ValueError("ROC-AUC requires both classes present in y_true")
    order         = np.argsort(scores, kind="mergesort")
    sorted_scores = scores[order]
    starts        = np.r_[0, np.flatnonzero(np.diff(sorted_scores)) + 1]
    ends          = np.r_[starts[1:], len(scores)]
    sorted_ranks  = np.repeat((starts + 1 + ends) / 2.0, ends - starts)
    ranks         = np.empty(len(scores), dtype=np.float64)
    ranks[order]  = sorted_ranks
    return float((ranks[y_true == 1].sum() - pos * (pos + 1) / 2.0) / (pos * neg))


In [7]:
print("Computing individual OOF AUCs ...")
individual_aucs = {}
for key, oof_arr in oof_dict.items():
    individual_aucs[key] = binary_roc_auc(y, oof_arr)

ordered_models = [
    name
    for name, _ in sorted(
        individual_aucs.items(), key=lambda kv: kv[1], reverse=True
    )
]

print(f"\n{'Rank':<5} {'OOF AUC':>12}  Model key")
print("-" * 80)
for rank, name in enumerate(ordered_models, 1):
    print(f"{rank:<5} {individual_aucs[name]:>12.8f}  {name}")


Computing individual OOF AUCs ...

Rank       OOF AUC  Model key
--------------------------------------------------------------------------------
1       0.96872158  solution_5/oof/oof_naji04.npy
2       0.96861856  solution_5/oof/oof_naji02.npy
3       0.96860117  solution_2/oof_catnative.npy
4       0.96852613  solution_5/oof/oof_lookup.npy
5       0.96848725  solution_5/oof/oof_tabm_x12.npy
6       0.96845113  solution_2/oof_gxgbcs4.npy
7       0.96843645  solution_5/oof/oof_pub_rmlp.npy
8       0.96840507  solution_2/oof_gcatnote.npy
9       0.96837200  solution_2/oof_xgbte.npy
10      0.96836915  solution_5/oof/oof_tabm_bounds.npy
11      0.96830178  solution_2/oof_gxgbd8.npy
12      0.96830134  solution_2/oof_glgbcs3.npy
13      0.96827470  solution_2/oof_gcatlr02.npy
14      0.96819320  solution_2/oof_glgbd4.npy
15      0.96816458  solution_2/oof_lgbte.npy
16      0.96815473  solution_2/oof_lgbs7.npy
17      0.96809329  solution_2/oof_gcatseed7.npy
18      0.96809112  solution_2

In [ ]:
print(f"Using PyTorch device : {DEVICE}")
print()

t0 = time.perf_counter()

# Convert all OOF arrays to torch tensors on the target device
oof_tensors = {
    name: torch.as_tensor(oof_dict[name], dtype=torch.float64, device=DEVICE)
    for name in ordered_models
}

# Weight grid: 0.01 ... 0.99 (99 steps) — same as reference notebook
weights_pool = torch.linspace(0.01, 0.99, 99, dtype=torch.float64, device=DEVICE)
weight_grid  = weights_pool[:, None]   # (99, 1) broadcasts over N sample rows

# Initialise with the single best model
base_model       = ordered_models[0]
best_cv          = individual_aucs[base_model]
running_ensemble = oof_tensors[base_model].clone()
selected_models  = [base_model]
final_weights    = {base_model: 1.0}

print(f"Starting model : {base_model}")
print(f"OOF AUC        : {best_cv:.8f}")
print()

for model_name in ordered_models:
    if model_name == base_model:
        continue

    pred = oof_tensors[model_name]

    # All 99 candidate blends in one vectorised op — shape (99, N)
    trials = (
        (1.0 - weight_grid) * running_ensemble[None, :]
        + weight_grid * pred[None, :]
    )

    # Score every candidate on CPU (numpy AUC function)
    trials_np = trials.cpu().numpy()   # (99, N)
    trial_scores = [
        binary_roc_auc(y, trials_np[i])
        for i in range(len(weights_pool))
    ]

    local_idx   = int(np.argmax(trial_scores))
    local_score = float(trial_scores[local_idx])

    if local_score <= best_cv:
        print(f"[SKIP] {model_name}")
        continue

    w = float(weights_pool[local_idx].item())
    print(
        f"[ADD ] {model_name:<55}"
        f" weight={w:.2f}  {best_cv:.8f} -> {local_score:.8f}"
    )

    running_ensemble = trials[local_idx]   # keep on device for next iteration
    best_cv          = local_score
    selected_models.append(model_name)

    # Dilute existing model weights by (1 - w)
    for k in final_weights:
        final_weights[k] *= (1.0 - w)
    final_weights[model_name] = w

# Normalise (floating-point safety)
total = sum(final_weights.values())
final_weights = {k: v / total for k, v in final_weights.items()}

elapsed = time.perf_counter() - t0
print("\n" + "-" * 60)
print(f"Hill-climbing complete in {elapsed:.1f}s")
print(f"Selected {len(selected_models)} / {len(ordered_models)} models")
print(f"Final ensemble OOF AUC : {best_cv:.8f}")


Using PyTorch device : cpu

Starting model : solution_5/oof/oof_naji04.npy
OOF AUC        : 0.96872158

[ADD ] solution_5/oof/oof_naji02.npy                           weight=0.18  0.96872158 -> 0.96872677
[ADD ] solution_2/oof_catnative.npy                            weight=0.47  0.96872677 -> 0.96916864
[ADD ] solution_5/oof/oof_lookup.npy                           weight=0.35  0.96916864 -> 0.96941342
[ADD ] solution_5/oof/oof_tabm_x12.npy                         weight=0.19  0.96941342 -> 0.96947121
[ADD ] solution_2/oof_gxgbcs4.npy                              weight=0.06  0.96947121 -> 0.96947588
[SKIP] solution_5/oof/oof_pub_rmlp.npy
[ADD ] solution_2/oof_gcatnote.npy                             weight=0.03  0.96947588 -> 0.96947679
[ADD ] solution_2/oof_xgbte.npy                                weight=0.01  0.96947679 -> 0.96947694
[SKIP] solution_5/oof/oof_tabm_bounds.npy
[SKIP] solution_2/oof_gxgbd8.npy
[SKIP] solution_2/oof_glgbcs3.npy
[SKIP] solution_2/oof_gcatlr02.npy
[SKIP]

In [ ]:
# ── Build test-probability blend ──────────────────────────────────────────────────
test_blend = np.zeros(n_test, dtype=np.float64)
for name, weight in final_weights.items():
    test_blend += weight * test_dict[name]

# ── Save blended test probabilities (.npy) ─────────────────────────────────────
blend_npy_path = ARTIFACTS_DIR / "hill_climbing_test_probabilities.npy"
np.save(blend_npy_path, test_blend)
print(f"Saved test probabilities -> {blend_npy_path}")

# ── Save submission CSV ───────────────────────────────────────────────────────────────
submission_path = ARTIFACTS_DIR / "hill_climbing_submission.csv"
pd.DataFrame(
    {"id": test_ids, TARGET_COL: test_blend}
).to_csv(submission_path, index=False)
print(f"Saved submission CSV    -> {submission_path}")

# ── Save result report (JSON) ─────────────────────────────────────────────────────────
result = {
    "metric": "roc_auc",
    "device": str(DEVICE),
    "total_candidates": len(ordered_models),
    "selected_model_count": len(selected_models),
    "selected_models": selected_models,
    "final_oof_auc": best_cv,
    "individual_oof_aucs": {
        k: round(v, 10)
        for k, v in sorted(
            individual_aucs.items(), key=lambda kv: kv[1], reverse=True
        )
    },
    "weights": dict(
        sorted(final_weights.items(), key=lambda kv: kv[1], reverse=True)
    ),
}

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
run_dir = ARTIFACTS_DIR / timestamp
run_dir.mkdir(parents=True, exist_ok=True)
result_json_path = run_dir / "hill_climbing_result.json"
result_json_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
print(f"Saved result report     -> {result_json_path}")

# ── Human-readable weight summary ───────────────────────────────────────────────────
print("\n" + "-" * 60)
print(f"FINAL WEIGHTS  (normalised, sum = {sum(final_weights.values()):.6f})")
print("-" * 60)
for model, w in sorted(
    final_weights.items(), key=lambda kv: kv[1], reverse=True
):
    bar = '#' * int(w * 40)
    print(f"{w:6.4f}  {bar:<40}  {model}")

print(f"\nTest prob range : [{test_blend.min():.6f}, {test_blend.max():.6f}]")
print(f"Test prob mean  : {test_blend.mean():.6f}")


Saved test probabilities -> ensemble_artifacts\hill_climbing_test_probabilities.npy
Saved submission CSV    -> ensemble_artifacts\hill_climbing_submission.csv
Saved result report     -> ensemble_artifacts\hill_climbing_result.json

------------------------------------------------------------
FINAL WEIGHTS  (normalised, sum = 1.000000)
------------------------------------------------------------
0.2196  ########                                  solution_5/oof/oof_lookup.npy
0.1917  #######                                   solution_2/oof_catnative.npy
0.1772  #######                                   solution_5/oof/oof_naji04.npy
0.1472  #####                                     solution_5/oof/oof_tabm_x12.npy
0.0548  ##                                        solution_5/oof/oof_latr1_xgb.npy
0.0494  #                                         solution_2/oof_gxgbcs4.npy
0.0389  #                                         solution_5/oof/oof_naji02.npy
0.0388  #                                

## Step 7 Meta Stacking

### Step 7a — Meta-Stacking: Build Feature Matrix

Transforms the OOF/test binary probability arrays into logit-space features,  
then concatenates them into a 2-D meta-feature matrix `X_train` / `X_test`.  
This mirrors the exact logic from `reference_scrips/meta_stacking.py`.


In [9]:
# ============================================================
# Meta Stacking with an Optimised PyTorch Tabular MLP
# Adapted from reference_scrips/meta_stacking.py
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ── Validate that data is available ──────────────────────────────────────────
model_names = sorted(oof_dict.keys())
if not model_names:
    raise RuntimeError("No OOF predictions were loaded; cannot build a meta stack.")
missing_test = [name for name in model_names if name not in test_dict]
if missing_test:
    raise RuntimeError(f"Missing test probabilities for: {missing_test}")

print(f"Base models available : {len(model_names)}")


def probability_to_logits(prob_array, epsilon=1e-7):
    """
    Stabilises and transforms probabilities into log-odds (logits).
    Works for both 1-D (binary) and 2-D (multiclass) probability arrays.
    """
    prob_clipped = np.clip(prob_array, epsilon, 1.0 - epsilon)
    return np.log(prob_clipped / (1.0 - prob_clipped))


# ── Build OOF meta-feature matrix ────────────────────────────────────────────
# Each model contributes a 1-D binary probability -> column of logits
X_train_list = []
for name in model_names:
    raw_probs = oof_dict[name].astype(np.float32)
    logit_features = probability_to_logits(raw_probs).reshape(-1, 1)
    X_train_list.append(logit_features)

X_train_meta = np.concatenate(X_train_list, axis=1)   # shape: (n_train, n_models)

# ── Build Test meta-feature matrix ───────────────────────────────────────────
X_test_list = []
for name in model_names:
    raw_test_probs = test_dict[name].astype(np.float32)
    logit_test_features = probability_to_logits(raw_test_probs).reshape(-1, 1)
    X_test_list.append(logit_test_features)

X_test_meta = np.concatenate(X_test_list, axis=1)     # shape: (n_test, n_models)

print(f"X_train_meta shape : {X_train_meta.shape}")
print(f"X_test_meta  shape : {X_test_meta.shape}")


Using device: cpu
Base models available : 89
X_train_meta shape : (691369, 89)
X_test_meta  shape : (296302, 89)


### Step 7b — Meta-Stacking: Define TabularMLP & Train 5-Fold CV

Trains a `TabularMLP` meta-learner with 5-fold stratified CV on the logit meta-features.  
Uses **ROC-AUC** on the positive-class probability as the validation metric.  
Accumulates OOF predictions and blended test predictions (averaged across folds).


In [10]:
# ── TabularMLP definition (exact architecture from reference script) ──────────
class TabularMLP(nn.Module):
    """Two-hidden-layer MLP with LayerNorm and Dropout for meta-stacking."""

    def __init__(self, input_dim: int, num_classes: int):
        super().__init__()
        # LayerNorm since inputs are concatenated logit-transformed probabilities
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.30),   # slightly higher dropout to prevent meta-overfitting
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


# ── Hyper-parameters (from reference script) ─────────────────────────────────
NUM_CLASSES = 2       # binary: not-addicted / addicted
EPOCHS      = 35      # enough epochs for convergence
BATCH_SIZE  = 1024    # smaller batch for better gradient granularity
VERBOSE     = 2       # print every N epochs

# ── 5-fold stratified CV ─────────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_meta         = np.zeros((len(y), NUM_CLASSES), dtype=np.float32)
test_preds_folds = np.zeros((len(X_test_meta), NUM_CLASSES), dtype=np.float32)

X_test_t = torch.tensor(X_test_meta, dtype=torch.float32, device=DEVICE)

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_train_meta, y)):
    print(f"\n--- Training Fold {fold + 1} ---")

    X_tr = torch.tensor(X_train_meta[tr_idx], dtype=torch.float32, device=DEVICE)
    y_tr = torch.tensor(y[tr_idx],            dtype=torch.long,    device=DEVICE)
    X_va = torch.tensor(X_train_meta[va_idx], dtype=torch.float32, device=DEVICE)
    y_va = torch.tensor(y[va_idx],            dtype=torch.long,    device=DEVICE)

    # Class-balanced loss weights
    counts        = torch.bincount(y_tr)
    class_weights = counts.sum() / (len(counts) * counts.float())
    criterion     = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))

    model     = TabularMLP(input_dim=X_tr.shape[1], num_classes=NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS, eta_min=1e-5
    )

    train_ds = TensorDataset(X_tr, y_tr)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

    best_score = -1.0
    best_state = None

    for epoch in range(EPOCHS):
        # ── Training pass ────────────────────────────────────────────────────
        model.train()
        for X_batch, y_batch in train_dl:
            optimizer.zero_grad()
            logits = model(X_batch)
            loss   = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
        scheduler.step()

        # ── Validation pass ───────────────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            val_logits = model(X_va)
            val_pred   = torch.argmax(val_logits, dim=1)
            # Use ROC-AUC on positive-class softmax probability
            val_proba = val_logits.softmax(dim=1)[:, 1].cpu().numpy()
            score = roc_auc_score(
                y_va.cpu().numpy(), val_proba
            )

        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())

        if epoch % VERBOSE == 0 or epoch == EPOCHS - 1:
            print(f"  Epoch {epoch:3d} | ROC-AUC: {score:.5f}")

    print(f"Fold {fold + 1} Best ROC-AUC: {best_score:.5f}")

    # ── Load best checkpoint for OOF & test inference ────────────────────────
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        oof_meta[va_idx]   = model(X_va).softmax(dim=1).cpu().numpy()
        # Blend test predictions across all fold models (average)
        test_preds_folds  += model(X_test_t).softmax(dim=1).cpu().numpy() / cv.n_splits

# ── Overall CV score ─────────────────────────────────────────────────────────
meta_score = roc_auc_score(y, oof_meta[:, 1])
print(f"\nMeta stacking CV ROC-AUC: {meta_score:.6f}")



--- Training Fold 1 ---
  Epoch   0 | ROC-AUC: 0.96865
  Epoch   2 | ROC-AUC: 0.96888
  Epoch   4 | ROC-AUC: 0.96895
  Epoch   6 | ROC-AUC: 0.96896
  Epoch   8 | ROC-AUC: 0.96904
  Epoch  10 | ROC-AUC: 0.96906
  Epoch  12 | ROC-AUC: 0.96906
  Epoch  14 | ROC-AUC: 0.96909
  Epoch  16 | ROC-AUC: 0.96911
  Epoch  18 | ROC-AUC: 0.96909
  Epoch  20 | ROC-AUC: 0.96908
  Epoch  22 | ROC-AUC: 0.96913
  Epoch  24 | ROC-AUC: 0.96913
  Epoch  26 | ROC-AUC: 0.96913
  Epoch  28 | ROC-AUC: 0.96913
  Epoch  30 | ROC-AUC: 0.96913
  Epoch  32 | ROC-AUC: 0.96913
  Epoch  34 | ROC-AUC: 0.96913
Fold 1 Best ROC-AUC: 0.96914

--- Training Fold 2 ---
  Epoch   0 | ROC-AUC: 0.96937
  Epoch   2 | ROC-AUC: 0.96964
  Epoch   4 | ROC-AUC: 0.96967
  Epoch   6 | ROC-AUC: 0.96976
  Epoch   8 | ROC-AUC: 0.96971
  Epoch  10 | ROC-AUC: 0.96976
  Epoch  12 | ROC-AUC: 0.96982
  Epoch  14 | ROC-AUC: 0.96983
  Epoch  16 | ROC-AUC: 0.96981
  Epoch  18 | ROC-AUC: 0.96983
  Epoch  20 | ROC-AUC: 0.96983
  Epoch  22 | ROC-AUC:

### Step 7c — Meta-Stacking: Save Artifacts

Saves all outputs to `ensemble_artifacts/<YYYY-MM-DD_HH-MM-SS>_meta_sol/`:
- `meta_sol_oof_probabilities.npy` — OOF softmax probabilities shape (N, 2)
- `meta_sol_test_probabilities.npy` — blended test softmax probabilities shape (N_test, 2)
- `meta_sol_submission.csv` — final submission file
- `meta_sol_result.json` — CV score, model list, hyper-parameters


In [11]:
# ── Timestamped output directory ─────────────────────────────────────────────
timestamp    = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
meta_run_dir = ARTIFACTS_DIR / f"{timestamp}_meta_sol"
meta_run_dir.mkdir(parents=True, exist_ok=True)

# ── 1. OOF probabilities ──────────────────────────────────────────────────────
oof_npy_path = meta_run_dir / "meta_sol_oof_probabilities.npy"
np.save(oof_npy_path, oof_meta)
print(f"Saved OOF probabilities    -> {oof_npy_path}")

# ── 2. Test probabilities ─────────────────────────────────────────────────────
test_npy_path = meta_run_dir / "meta_sol_test_probabilities.npy"
np.save(test_npy_path, test_preds_folds)
print(f"Saved test probabilities   -> {test_npy_path}")

# ── 3. Submission CSV ─────────────────────────────────────────────────────────
# Positive class probability (class=1) used as the submission score
submission_path = meta_run_dir / "meta_sol_submission.csv"
pd.DataFrame(
    {"id": test_ids, TARGET_COL: test_preds_folds[:, 1]}
).to_csv(submission_path, index=False)
print(f"Saved submission CSV       -> {submission_path}")

# ── 4. Result JSON ────────────────────────────────────────────────────────────
meta_result = {
    "method"          : "meta_stacking_tabular_mlp",
    "timestamp"       : timestamp,
    "device"          : str(DEVICE),
    "metric"          : "roc_auc",
    "cv_score"        : round(float(meta_score), 8),
    "num_classes"     : NUM_CLASSES,
    "n_folds"         : cv.n_splits,
    "epochs"          : EPOCHS,
    "batch_size"      : BATCH_SIZE,
    "base_model_count": len(model_names),
    "base_models"     : model_names,
    "architecture": {
        "layers"   : [
            "Linear(input->256)", "LayerNorm(256)", "ReLU", "Dropout(0.30)",
            "Linear(256->128)",   "LayerNorm(128)", "ReLU", "Dropout(0.20)",
            "Linear(128->64)",    "ReLU",
            "Linear(64->num_classes)"
        ],
        "optimizer": "AdamW(lr=2e-3, weight_decay=1e-4)",
        "scheduler": "CosineAnnealingLR(T_max=EPOCHS, eta_min=1e-5)",
        "loss"     : "CrossEntropyLoss(class_weighted)",
    },
    "artifacts": {
        "oof_probabilities" : str(oof_npy_path),
        "test_probabilities": str(test_npy_path),
        "submission_csv"    : str(submission_path),
    },
}

result_json_path = meta_run_dir / "meta_sol_result.json"
result_json_path.write_text(json.dumps(meta_result, indent=2), encoding="utf-8")
print(f"Saved result JSON          -> {result_json_path}")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n==================================================")
print("Meta Stacking Submission Generated")
print("==================================================")
print(f"CV ROC-AUC            : {meta_score:.6f}")
print(f"Base models used      : {len(model_names)}")
print(f"Output directory      : {meta_run_dir}")
print(f"Test prob range       : [{test_preds_folds[:, 1].min():.6f}, {test_preds_folds[:, 1].max():.6f}]")
print(f"Test prob mean        : {test_preds_folds[:, 1].mean():.6f}")


Saved OOF probabilities    -> ensemble_artifacts\2026-08-22_17-48-01_meta_sol\meta_sol_oof_probabilities.npy
Saved test probabilities   -> ensemble_artifacts\2026-08-22_17-48-01_meta_sol\meta_sol_test_probabilities.npy
Saved submission CSV       -> ensemble_artifacts\2026-08-22_17-48-01_meta_sol\meta_sol_submission.csv
Saved result JSON          -> ensemble_artifacts\2026-08-22_17-48-01_meta_sol\meta_sol_result.json

Meta Stacking Submission Generated
CV ROC-AUC            : 0.969614
Base models used      : 89
Output directory      : ensemble_artifacts\2026-08-22_17-48-01_meta_sol
Test prob range       : [0.000851, 0.999969]
Test prob mean        : 0.653437
